In [63]:
from pydantic import BaseModel,field_validator
import re
import json 

In [64]:
class Rating(BaseModel):
    value:float
    count:int
    review_count:int

class Price(BaseModel):
    price:float
    currency:str

class Provider(BaseModel):
    name:str
    product_rating:float
    service_rating:float
    experience:str

class Product(BaseModel):
    brand:str
    name:str
    description:str
    image:list
    rating:Rating
    price:Price
    provider:Provider
    highlights:dict
    policy:str
    manufacturer_info:str
    

In [70]:
json_file='flipkart_data.json'
with open(json_file,'r',encoding='utf-8') as f:
    data=json.load(f)

In [ ]:
general_specs = {}

path_0 = data["multiWidgetState"]["widgetsData"]["slots"]

try:
    grid_data = []
    
    for slot in path_0:
        try:
            if "slotData" in slot and "widget" in slot["slotData"] and "data" in slot["slotData"]["widget"]:
                dlsData = slot["slotData"]["widget"]["data"]["dlsData"]
                
                if "product-details-grid_0" in dlsData:
                    grid_data = dlsData["product-details-grid_0"]["value"]["gridData_0"]["value"]
                    break
                
                # if "rpd_tab_showcase_vertical_list_0" in dlsData:
                #     specs_grid = dlsData["rpd_tab_showcase_vertical_list_0"]["value"]["rpd_specifications_grid_layout_1"]["value"]
                #     if "gridData_0" in specs_grid:
                #         grid_data = specs_grid["gridData_0"]["value"]
                #         break

        except (Exception):
            continue
    
    if grid_data:
        for item in grid_data:
            item_value = item["value"]
            
            if "label_0" in item_value and "label_1" in item_value:
                key = item_value["label_0"]["value"]["text"]
                value = item_value["label_1"]["value"]["text"]
                
                if len(value) > 0:
                    value = value[0]
                
                general_specs[key] = value
    
except Exception as e:
    print(f"Error extracting specifications: {e}")
    print("general_specs will be empty")


In [72]:
product_path=data['multiWidgetState']['pageDataResponse']['seoData']['schema']

products_data = []

for item in product_path:
    brand = item['brand']['name']
    name = item['name']
    description = item['description']
    image = item['image']
    rating_path = item['aggregateRating']
    
    rating_data = Rating(
         value=rating_path['ratingValue'],
         count=rating_path['ratingCount'],
         review_count=rating_path['reviewCount']
    ) 

    price_path = item['offers']
    price_data = Price(
         price=price_path['price'],
         currency=price_path['priceCurrency']
    )
    
    experience = "Not Available"
    service_rating = 0
    fulfilled_by = "Not Available"
    
    try:
        path_1 = None
        slots = data['multiWidgetState']['widgetsData']['slots']
        
        for slot in slots:
            try:
                if "slotData" in slot and "widget" in slot["slotData"] and "data" in slot["slotData"]["widget"]:
                    dlsData = slot["slotData"]["widget"]["data"]["dlsData"]
                    
                    if "default_fk_pp_delivery_widget_seller_2" in dlsData:
                        path_1 = dlsData
                        break
            except (Exception):
                continue
        
        if path_1 is None:
            raise Exception("Could not find seller details widget")
        
        path_2 = path_1['default_fk_pp_delivery_widget_seller_2']['value']
        fulfilled_by = path_2['label_0']['value']['text']
        
        service_rating = float(path_2['label_1']['value']['text'])
        
        experience = path_2['label_3']['value']['text']
        
    except Exception as e:
        print(f"Could not extract seller details: {e}")
        fulfilled_by = "Unknown Seller"
        experience = "Not Available"
        service_rating = 0
    
    highlights = {}
    highlights.update(general_specs)
    
    policy = item['offers']['hasMerchantReturnPolicy']['description']
    
    manufacturer_info = brand
    
    provider_data = Provider(
         name=fulfilled_by,
         product_rating=rating_data.value,
         service_rating=service_rating,
         experience=experience
    )
    
    product_dict = Product(
         brand=brand,
         name=name,
         description=description,
         image=image,
         rating=rating_data,
         price=price_data,
         provider=provider_data,
         highlights=highlights,
         policy=policy,
         manufacturer_info=manufacturer_info
    )
    products_data.append(product_dict)

  


In [73]:
products_data

[Product(brand='Amul', name='Amul Gold Milk', description='Buy Amul Gold Milk for Rs.33.0 online. Amul Gold Milk at best prices with FREE shipping & cash on delivery. Only Genuine Products. 30 Day Replacement Guarantee.', image=['https://rukmini1.flixcart.com/image/1500/1500/xif0q/milk/v/y/k/500-gold-pouch-plain-amul-original-imah2hrgfkuanznh.jpeg?q=70', 'https://rukmini1.flixcart.com/image/1500/1500/xif0q/milk/p/m/r/500-gold-pouch-plain-amul-original-imah2hrgja3gpugy.jpeg?q=70', 'https://rukmini1.flixcart.com/image/1500/1500/xif0q/milk/6/k/e/500-gold-pouch-plain-amul-original-imah2hrg2gy2myxq.jpeg?q=70', 'https://rukmini1.flixcart.com/image/1500/1500/xif0q/milk/i/p/p/500-gold-pouch-plain-amul-original-imah2hrgxw7p7b9f.jpeg?q=70', 'https://rukmini1.flixcart.com/image/1500/1500/xif0q/milk/f/l/j/500-gold-pouch-plain-amul-original-imah2hrgqfakmgds.jpeg?q=70'], rating=Rating(value=4.5, count=1732, review_count=155), price=Price(price=33.0, currency='INR'), provider=Provider(name='Fulfilled

In [74]:

with open("output2.json", "w", encoding="utf-8") as f:
    json.dump([i.model_dump() for i in products_data], f, ensure_ascii=False, indent=4)
